# PaceAlgo ML — Notebook 07: Experiment Battery

**Zweck:** Bevor wir 3-4 Wochen in Funding/OI investieren, durchlaufen wir alle datengetriebenen Quick-Wins systematisch.

**7 Experimente — jedes mit gleichem Train/Val/Test-Split, gleichen Hyperparams, dokumentiertem Outcome:**

| # | Name | Was | Hypothese |
|---|---|---|---|
| 0 | Baseline (NB05) | Alles wie gehabt | Reference PF = 1.10 |
| 1 | Active features only | 16 statt 37 Features (dead 21 entfernt) | Weniger Noise → besseres Lernen |
| 2 | R=2.0 | Größeres R-Verhältnis, mehr Headroom | Baseline-WR 33% → ML kann 45% pushen → PF 1.5 |
| 3 | Crypto-only | Nur BTC, ETH trainiert | Crypto braucht eigene Patterns |
| 4 | FX+Gold-only | Nur EUR, JPY, XAU | FX zeigte besseren Hold-out (PF 1.22 GBPUSD) |
| 5 | Top-10 SHAP | Nur Top 10 Features nach SHAP | Maximale Fokussierung |
| 6 | Top-15 SHAP | Nur Top 15 Features nach SHAP | Mittelweg |

**Output:** Comparison-Table mit OOS TEST PF, WR, Trade-Count.

**Laufzeit:** ~35-50 Min (7 Modelle × ~5 Min).

## 1. Environment Setup

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_PROCESSED = Path('/content/processed')
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    REPORTS_DIR = ARTIFACTS / 'reports'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED.glob('*.parquet'))) < (len(list(DRIVE_BACKUP.glob('*.parquet'))) if DRIVE_BACKUP.exists() else 0):
        print('Restoring features from Drive backup...')
        !rsync -ah {DRIVE_BACKUP}/ {DATA_PROCESSED}/
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_PROCESSED, ARTIFACTS_REPORTS
    REPORTS_DIR = ARTIFACTS_REPORTS

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
print(f'Data: {DATA_PROCESSED}')

In [ ]:
!pip install -q lightgbm 2>&1 | tail -1

import pandas as pd
import numpy as np
import lightgbm as lgb
from datetime import datetime

from core.config import (
    CRYPTO_SYMBOLS, FX_SYMBOLS, METAL_SYMBOLS,
    PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END,
)
from core.train import (
    stack_symbols, get_feature_columns,
    walk_forward_split, binary_label_for_long,
    train_lgbm, trading_metrics_from_predictions, sweep_threshold,
)
from core.analysis import confidence_percentile_sweep

ALL_SYMBOLS = CRYPTO_SYMBOLS + FX_SYMBOLS + METAL_SYMBOLS
print(f'All symbols: {ALL_SYMBOLS}')
print(f'Dev hold-out: {DEV_HOLDOUT_SYMBOLS}')

## 2. Feature Subsets (from NB 06 SHAP analysis)

Ranking aus SHAP `mean_abs_shap` auf Test-Set vom NB 06:

In [ ]:
# Top features by SHAP mean_abs (from NB 06 output)
SHAP_RANKED = [
    'hour_sin',                       # 1
    'dist_to_swing_low_atr',          # 2
    'htf_1h_rsi_14',                  # 3
    'realized_vol_20',                # 4
    'atr_percentile_100',             # 5
    'atr_pct',                        # 6
    'dist_to_swing_high_atr',         # 7
    'volume_z_score',                 # 8
    'ema_20_slope_atr',               # 9
    'hour_cos',                       # 10
    'momentum_composite',             # 11
    'rvol_20',                        # 12
    'adx_14',                         # 13
    'ema_20_dist_atr',                # 14
    'htf_1h_atr_percentile_100',      # 15
    'adx_slope',                      # 16
]

TOP_10 = SHAP_RANKED[:10]
TOP_15 = SHAP_RANKED[:15]
ACTIVE = SHAP_RANKED  # all 16 active

print(f'TOP_10  ({len(TOP_10)}): {TOP_10}')
print(f'\nTOP_15  ({len(TOP_15)}): {TOP_15}')
print(f'\nACTIVE  ({len(ACTIVE)}): same as TOP_15 + adx_slope')

## 3. Shared Hyperparameters + Train-Eval Helper

In [ ]:
PARAMS = {
    'objective':        'binary',
    'metric':           'binary_logloss',
    'num_leaves':       15,
    'max_depth':        4,
    'min_data_in_leaf': 100,
    'learning_rate':    0.03,
    'num_iterations':   400,
    'lambda_l2':        0.5,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq':     5,
    'is_unbalance':     True,
    'verbose':          -1,
    'n_jobs':           -1,
}

def run_experiment(name, features_to_use=None, R=1.5, symbol_set=None):
    """Train + evaluate one experiment. Returns metrics dict."""
    print(f'\n{"="*70}')
    print(f'EXPERIMENT: {name}')
    print(f'  R={R}, symbols={symbol_set}, features={len(features_to_use) if features_to_use else "all"}')
    print('=' * 70)

    symbols = symbol_set if symbol_set else ALL_SYMBOLS

    # Load stacked data
    df = stack_symbols(DATA_PROCESSED, symbols=symbols, tfs=PRIMARY_TIMEFRAMES,
                        R=R, drop_holdout=DEV_HOLDOUT_SYMBOLS)
    if df.empty:
        print('  ERROR: empty stacked DataFrame')
        return None

    # Feature column selection
    if features_to_use is None:
        features_to_use = get_feature_columns(df)
    # Ensure all requested features exist
    available = set(df.columns)
    missing = [f for f in features_to_use if f not in available]
    if missing:
        print(f'  WARN: missing features: {missing}')
        features_to_use = [f for f in features_to_use if f in available]

    df_clean = df.dropna(subset=features_to_use)
    train_df, val_df, test_df = walk_forward_split(df_clean, TRAIN_END, VAL_END)
    print(f'  train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')

    X_train = train_df[features_to_use]; y_train = binary_label_for_long(train_df['label'])
    X_val = val_df[features_to_use]; y_val = binary_label_for_long(val_df['label'])
    X_test = test_df[features_to_use]; y_test = binary_label_for_long(test_df['label'])

    # Train
    model = train_lgbm(X_train, y_train, X_val, y_val, params=PARAMS, early_stopping_rounds=30)
    print(f'  trees: {model.num_trees()}, best_iter: {model.best_iteration}')

    # Threshold optimization on VAL
    val_proba = model.predict(X_val)
    sweep = sweep_threshold(val_df['label'], val_proba, tp_R=R)
    valid = sweep[sweep['n_trades'] >= 500]
    if len(valid) > 0:
        best_t = float(valid.loc[valid['profit_factor'].idxmax(), 'threshold'])
        best_val_pf = float(valid.loc[valid['profit_factor'].idxmax(), 'profit_factor'])
    else:
        best_t = 0.40
        best_val_pf = 0.0

    # Apply to TEST
    test_proba = model.predict(X_test)
    test_metrics = trading_metrics_from_predictions(test_df['label'], test_proba, best_t, tp_R=R)

    # Top-1% confidence
    sweep_pct = confidence_percentile_sweep(test_df['label'], test_proba, tp_R=R, percentiles=[1, 5, 10])
    top1 = sweep_pct[sweep_pct['top_pct'] == 1].iloc[0]
    top5 = sweep_pct[sweep_pct['top_pct'] == 5].iloc[0]
    top10 = sweep_pct[sweep_pct['top_pct'] == 10].iloc[0]

    result = {
        'experiment': name,
        'features_n': len(features_to_use),
        'R': R,
        'symbols': ','.join(symbols),
        'n_symbols': len(symbols),
        'n_train': len(train_df),
        'n_test': len(test_df),
        'best_threshold': best_t,
        'val_pf': best_val_pf,
        'test_pf_threshold': float(test_metrics['profit_factor']),
        'test_wr_threshold': float(test_metrics['win_rate']),
        'test_n_trades': int(test_metrics['n_trades']),
        'top1_pf': float(top1['profit_factor']),
        'top5_pf': float(top5['profit_factor']),
        'top10_pf': float(top10['profit_factor']),
        'trees': model.num_trees(),
    }

    print(f'  -> TEST PF (@thr {best_t:.2f}): {result["test_pf_threshold"]:.3f}, '
          f'top-1% PF: {result["top1_pf"]:.3f}, top-5% PF: {result["top5_pf"]:.3f}')
    return result

print('Helper ready.')

## 4. Run All 7 Experiments

In [ ]:
results = []

# Exp 0: Baseline reference (all 37 features, R=1.5, all 5 training symbols)
results.append(run_experiment('0_baseline', features_to_use=None, R=1.5, symbol_set=ALL_SYMBOLS))

# Exp 1: Active features only (16 features)
results.append(run_experiment('1_active_features', features_to_use=ACTIVE, R=1.5, symbol_set=ALL_SYMBOLS))

# Exp 2: R=2.0
results.append(run_experiment('2_R2.0', features_to_use=None, R=2.0, symbol_set=ALL_SYMBOLS))

# Exp 3: Crypto-only (BTC, ETH — SOL is hold-out, excluded automatically)
results.append(run_experiment('3_crypto_only', features_to_use=None, R=1.5,
                                symbol_set=CRYPTO_SYMBOLS))

# Exp 4: FX+Gold-only (EUR, USDJPY, XAU — GBPUSD is hold-out)
results.append(run_experiment('4_fx_gold_only', features_to_use=None, R=1.5,
                                symbol_set=FX_SYMBOLS + METAL_SYMBOLS))

# Exp 5: Top 10 SHAP features
results.append(run_experiment('5_top10_shap', features_to_use=TOP_10, R=1.5,
                                symbol_set=ALL_SYMBOLS))

# Exp 6: Top 15 SHAP features
results.append(run_experiment('6_top15_shap', features_to_use=TOP_15, R=1.5,
                                symbol_set=ALL_SYMBOLS))

print('\n\nAll experiments done.')

## 5. Comparison Table

In [ ]:
results_df = pd.DataFrame([r for r in results if r is not None])
display_cols = ['experiment', 'features_n', 'R', 'n_symbols',
                 'val_pf', 'test_pf_threshold', 'test_wr_threshold', 'test_n_trades',
                 'top1_pf', 'top5_pf', 'top10_pf', 'best_threshold']
print('All experiments side-by-side:')
display(results_df[display_cols].round(4))

# Save
out_path = REPORTS_DIR / 'experiment_battery_results.parquet'
results_df.to_parquet(out_path)
print(f'\nSaved: {out_path}')

## 6. Decision Matrix

In [ ]:
import matplotlib.pyplot as plt

# Visual comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(results_df))
width = 0.25

ax1.bar(x - width, results_df['test_pf_threshold'], width, label='PF @ best threshold', color='#3a7bd5')
ax1.bar(x, results_df['top5_pf'], width, label='PF @ top 5% confidence', color='#5cb85c')
ax1.bar(x + width, results_df['top1_pf'], width, label='PF @ top 1% confidence', color='#f0ad4e')
ax1.axhline(1.0, linestyle='--', color='gray', alpha=0.6)
ax1.axhline(1.4, linestyle='--', color='green', alpha=0.6, label='PF=1.4 target')
ax1.set_xticks(x)
ax1.set_xticklabels(results_df['experiment'], rotation=30, ha='right')
ax1.set_ylabel('Profit Factor (OOS test)')
ax1.set_title('Per-experiment OOS PF comparison')
ax1.legend()
ax1.grid(alpha=0.3, axis='y')

ax2.bar(x, results_df['test_n_trades'], color='#3a7bd5')
ax2.set_xticks(x)
ax2.set_xticklabels(results_df['experiment'], rotation=30, ha='right')
ax2.set_ylabel('Trades on TEST')
ax2.set_title('Per-experiment trade count')
ax2.grid(alpha=0.3, axis='y')
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Ranked summary — best experiments first
ranking = results_df[['experiment', 'test_pf_threshold', 'top1_pf', 'top5_pf', 'test_n_trades']].copy()
ranking['best_pf'] = ranking[['test_pf_threshold', 'top1_pf', 'top5_pf']].max(axis=1)
ranking = ranking.sort_values('best_pf', ascending=False).reset_index(drop=True)
print('Experiments ranked by best PF (any slice):')
display(ranking.round(4))

print('\n' + '=' * 70)
print('RECOMMENDATION LOGIC')
print('=' * 70)
best = ranking.iloc[0]
if best['best_pf'] >= 1.5:
    print(f'STRONG EDGE FOUND: {best["experiment"]} with PF {best["best_pf"]:.3f}')
    print('-> Proceed to Pine export with this configuration as V1 base.')
elif best['best_pf'] >= 1.4:
    print(f'MODERATE EDGE: {best["experiment"]} with PF {best["best_pf"]:.3f}')
    print('-> Worth proceeding to Pine export. Consider also adding Funding/OI later.')
elif best['best_pf'] >= 1.25:
    print(f'WEAK EDGE: {best["experiment"]} with PF {best["best_pf"]:.3f}')
    print('-> Marginal. Either accept (with realistic marketing) or invest in features.')
else:
    print(f'NO MATERIAL EDGE: best PF {best["best_pf"]:.3f}')
    print('-> Architecture revisit required. OHLCV-only ceiling reached.')
    print('   Next investment: Funding/OI/SMC features or rule-based v2.6 as V1.')

---

Schick mir die Comparison-Tabelle (Section 5) und den Decision-Block (Section 6 Ende). Dann sehen wir welche Konfiguration der klare Gewinner ist — und ob wir damit zu Pine Export gehen oder ob wir nach Funding/OI greifen müssen.